# Augmentation API for all image data inputs

In [ ]:
%pip install albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 369.4/369.4 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 583.6/583.6 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 178.4 MB/s eta 0:00:00


In [ ]:
# All imports for augmentation

from scipy.ndimage import gaussian_filter
from PIL import Image
import matplotlib.image as mpimg
import albumentations as albu
import math
import random
import cv2
import numpy as np
import os

In [ ]:
# Sub-functions for augmentations (can ignore)

def random_compression(img_array):

  '''Creates random compression on an image for quality = 30/50/70/90%'''

  quality = random.choice([30, 50, 70, 90]) # Random compression 30/50/70/90%

  image_array = np.random.randint(0, 256, (1080, 1920, 3), dtype=np.uint8)

  # 1. Compress the NumPy array into an in-memory JPEG byte buffer
  # [IMWRITE_JPEG_QUALITY] controls quality from 0-100 (lower means more compression)
  success, encoded_buffer = cv2.imencode('.jpg', img_array, [cv2.IMWRITE_JPEG_QUALITY, quality])

  # 2. Convert buffer to a compressed byte string or standard array
  compressed_bytes = encoded_buffer.tobytes()

  # 3. Decompress back into a usable NumPy array whenever needed
  decompressed_array = cv2.imdecode(np.frombuffer(compressed_bytes, np.uint8), cv2.IMREAD_COLOR)

  return decompressed_array

def random_blur(img_array):

  '''Creates random gaussian blur on an image for standard deviation = 0.5/1.0/2.0'''

  sigma = random.choice([0.5, 1.0, 2.0])

  blurred_img = gaussian_filter(img_array, sigma=sigma, axes=(0, 1))

  return blurred_img

def centre_crop(img_array):

  '''Creates 80% centre crop on an image'''

  length, width, height = img_array.shape

  # 1. Create zero array for all black
  cropped_array = np.zeros_like(img_array)

  # 2. Apply fixed crop ratio
  crop_ratio = 0.8

  # 3. Find start and end limits of length and width
  crop_length, crop_width = int(crop_ratio * length), int(crop_ratio * width)
  length_start, length_end = (length - crop_length) // 2, length - (length - crop_length) // 2
  width_start, width_end = (width - crop_width) // 2, width - (width - crop_width) // 2


  cropped_img = img_array[length_start:length_end, width_start:width_end]

  # 4. Put cropped image onto zero array to leave initial dimensions unchanged
  cropped_array[length_start:length_end, width_start:width_end] = cropped_img

  return cropped_array


def random_colour_jitter(img_array):

  '''Applies random colour jitter to the input up to 20%'''

  random_brightness = random.randint(1, 20) / 100
  random_contrast = random.randint(1, 20) / 100
  random_saturation = random.randint(1, 20) / 100

  transform = albu.ColorJitter(brightness=random_brightness, contrast=random_contrast, saturation=random_saturation)

  image = transform(image=img_array)['image']

  return image

def random_noise(img_array):

  '''Applies random gaussian noise to the input for 0.02, 0.05, 0.10'''

  sigma = random.choice([0.02, 0.05, 0.10])

  noise = np.random.normal(0, math.sqrt(sigma), img_array.shape) # Set mean to 0

  noisy_image = img_array + noise

  noisy_image = np.clip(noisy_image, 0, 255).astype(np.uint8)

  return noisy_image

def random_resize(img_array):

  '''Applies random resize to the input, and upsized back to its original dimensions'''

  resize_ratio = random.choice([0.5, 0.25])
  length, width, height = img_array.shape

  downsize = albu.augmentations.geometric.resize.Resize(int(resize_ratio * length), int(resize_ratio * width))
  upsize = albu.augmentations.geometric.resize.Resize(length, width)

  downsized_image = downsize(image=img_array)['image']
  upsized_image = upsize(image=img_array)['image']

  return upsized_image

In [ ]:
# Main Function for augmentation of a singular image

def random_augment(img_array):
  '''
  Choose one of the random augmentations and applies it on a 3D NumPy Array Image

  Returns a 3D NumPy Array of the augmented images
  '''

  func_dict = {
      'Random Compression': random_compression,
      'Random Blur': random_blur,
      'Centre Crop': centre_crop,
      'Random Jitter': random_colour_jitter,
      'Random Noise': random_noise,
      'Random Resize': random_resize,
      'None': None
  }

  random_func = random.choice(list(func_dict.keys())) # Chooses a random function to apply

  if random_func == 'None': # 1/7 chance no augmentation is applied
    return img_array

  altered_img = func_dict[random_func](img_array)

  return np.array([altered_img]).squeeze()

# Integration of the Data Augmentation Function, as well as the existing CLIP trained model into the product pipeline

In [ ]:
!pip install -q open_clip_torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# All imports for the main Pipeline

from PIL import Image
import joblib, torch, open_clip, torch.nn as nn, torch.nn.functional as F
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import pandas as pd

## DCT Support Functions

In [ ]:
from scipy.fft import dctn

def stage1_2_block_dct_from_array(bgr):
    """
    Stage 1 (Ingestion) + Stage 2 (Mathematical prep) on an already-loaded BGR array.
    Returns: (h_blocks, w_blocks, 8, 8) array of Y-channel block DCT coefficients.
    """
    ycbcr = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)  # Y, Cr, Cb
    y = ycbcr[:, :, 0].astype(np.float32) - 128.0   # center like JPEG does

    h, w = y.shape
    hb, wb = h // 8, w // 8
    y = y[:hb * 8, :wb * 8]  # crop any leftover rows/cols that don't fill a full 8x8 block

    blocks = y.reshape(hb, 8, wb, 8).transpose(0, 2, 1, 3)
    blocks = dctn(blocks, type=2, norm='ortho', axes=(-2, -1))

    return blocks.astype(np.float32)


def stage3_forensic_scan(blocks):
    """
    blocks: (hb, wb, 8, 8) block DCT coefficients.
    Returns (coeff_stats, [upsampling_score, double_compress_score]).
    """
    hb, wb, _, _ = blocks.shape
    flat = blocks.reshape(hb, wb, 64)

    coeff_stats = flat.mean(axis=(0, 1))

    dc_grid = flat[:, :, 0]
    fft_mag = np.abs(np.fft.fft2(dc_grid))
    fft_mag[0, 0] = 0
    upsampling_score = fft_mag.max() / (fft_mag.mean() + 1e-8)

    ac1 = flat[:, :, 1].flatten()
    hist, _ = np.histogram(ac1, bins=16)
    hist_fft = np.abs(np.fft.fft(hist.astype(np.float32)))
    hist_fft[0] = 0
    double_compress_score = hist_fft.max() / (hist_fft.mean() + 1e-8)

    return coeff_stats, np.array([upsampling_score, double_compress_score], dtype=np.float32)

class CoeffResNet(nn.Module):

  def __init__(self, in_ch=64, emb_dim=128):
      super().__init__()
      self.conv1 = nn.Conv2d(in_ch, 128, 3, padding=1)
      self.bn1 = nn.BatchNorm2d(128)
      self.conv2 = nn.Conv2d(128, emb_dim, 3, padding=1)
      self.bn2 = nn.BatchNorm2d(emb_dim)
      self.head = nn.Linear(emb_dim, 1)

  def forward(self, x, return_embedding=False):
      x = F.relu(self.bn1(self.conv1(x)))
      x = F.relu(self.bn2(self.conv2(x)))
      emb = F.adaptive_avg_pool2d(x, 1).flatten(1)  # (B, emb_dim)
      if return_embedding:
          return emb
      return self.head(emb).squeeze(-1)

In [195]:
class TopLayer(nn.Module):
  def __init__(self,
               clip_model="/content/drive/MyDrive/techjam/standard_v2.joblib",
               cnn_model="/content/drive/MyDrive/techjam/efficientnet_b0_detector-2.pt",
               dct_model="/content/drive/MyDrive/techjam/dct_logreg.joblib",
               dct_coeff_model="/content/drive/MyDrive/techjam/dct_coeff_net.pth"):

    super().__init__()

    self.CHECKPOINT_DIR = "/content/drive/MyDrive/techjam"
    self.DCT_SIZE = 256  # must match the resize used in ForensicDataset during DCT-model training

    self.device = "cuda" if torch.cuda.is_available() else "cpu"

    self.clip = joblib.load(clip_model)
    self.cnn = self.build_efficientnet(freeze_until_last_n_blocks=1).eval()

    # DCT pipeline: sklearn clf (StandardScaler + LogisticRegression) + CoeffResNet embedding CNN
    self.dct = joblib.load(dct_model)
    self.dct_coeff_net = CoeffResNet(in_ch=64).to(self.device)
    self.dct_coeff_net.load_state_dict(torch.load(dct_coeff_model, map_location=self.device))
    self.dct_coeff_net.eval()

    self.model, _, self.clip_preprocess = open_clip.create_model_and_transforms(self.clip["clip_model"], pretrained=self.clip["clip_pretrained"])
    self.model = self.model.to(self.device).eval()

    self.cnn_transform = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

  def build_efficientnet(self, freeze_until_last_n_blocks=1):

    checkpoint = torch.load(
        os.path.join(self.CHECKPOINT_DIR, "efficientnet_b0_detector-2.pt"),
        map_location=self.device,
        weights_only=False,
    )

    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=1)

    for param in model.parameters():
        param.requires_grad = False
    for stage in model.blocks[-freeze_until_last_n_blocks:]:
        for param in stage.parameters():
            param.requires_grad = True
    for module in [model.conv_head, model.bn2, model.classifier]:
        for param in module.parameters():
            param.requires_grad = True
    model = model.to(self.device)

    model.load_state_dict(checkpoint["state_dict"])

    return model

  @torch.no_grad()
  def clip_predict(self, imgs, which="robust"):
      '''Batched: imgs is a list of HxWx3 uint8 arrays. Returns (B,) tensor of P(AI).'''
      if isinstance(imgs, np.ndarray) and imgs.ndim == 3:  # single-image fallback
          imgs = [imgs]
      # preprocess each (resizes to CLIP's fixed input) then stack -> one encoder pass
      batch = torch.stack([self.clip_preprocess(Image.fromarray(a)) for a in imgs]).to(self.device)
      f = F.normalize(self.model.encode_image(batch).float(), dim=-1).cpu().numpy()  # (B, D)
      probs = self.clip["probes"][which].predict_proba(f)[:, 1]                       # vectorised over B
      return torch.tensor(probs, dtype=torch.float32)


  def cnn_predict(self, imgs):
      '''Batched: imgs is a list of HxWx3 uint8 arrays. Returns (B,) tensor of P(AI).'''
      if isinstance(imgs, np.ndarray) and imgs.ndim == 3:
          imgs = [imgs]
      self.cnn.eval()
      with torch.no_grad():
          # A.Resize(224,224) makes every image the same shape -> stackable
          batch = torch.stack([self.cnn_transform(image=a)["image"] for a in imgs]).to(self.device)
          logits = self.cnn(batch).squeeze(1)          # one forward pass for all B
          probs = torch.sigmoid(logits).detach().cpu()
      return probs                                      # (B,)


  def dct_predict(self, paths):
      '''Batched: paths is a list of file-path strings. Returns (B,) tensor of P(AI).'''
      if isinstance(paths, str):
          paths = [paths]
      coeff_mats, forensic_list = [], []
      for p in paths:
          bgr = cv2.imread(p, cv2.IMREAD_COLOR)
          bgr = cv2.resize(bgr, (self.DCT_SIZE, self.DCT_SIZE), interpolation=cv2.INTER_AREA)
          blocks = stage1_2_block_dct_from_array(bgr)              # (hb, wb, 8, 8)
          _, forensic_scalars = stage3_forensic_scan(blocks)      # (2,)
          hb, wb, _, _ = blocks.shape
          coeff_mats.append(torch.tensor(blocks.reshape(hb, wb, 64)).permute(2, 0, 1).float())
          forensic_list.append(forensic_scalars)
      # fixed DCT_SIZE => every coeff map is (64, 32, 32) => stackable into one CNN pass
      batch = torch.stack(coeff_mats).to(self.device)             # (B, 64, 32, 32)
      with torch.no_grad():
          cnn_emb = self.dct_coeff_net(batch, return_embedding=True).cpu().numpy()  # (B, 128)
      feat = np.concatenate([cnn_emb, np.stack(forensic_list)], axis=1)             # (B, 130)
      probs = self.dct.predict_proba(feat)[:, 1]                                    # vectorised
      return torch.tensor(probs, dtype=torch.float32)

  def forward(self, paths):
    '''Accepts a single path (str) or a list of paths. Returns (B, 3) tensor:
       columns are [clip_prob, cnn_prob, dct_prob].'''
    if isinstance(paths, str):
        paths = [paths]

    img_arrs  = [np.array(Image.open(p).convert("RGB")) for p in paths]
    augmented = [random_augment(a) for a in img_arrs]   # per-image random aug, as before

    clip_probs = self.clip_predict(augmented)   # (B,)
    cnn_probs  = self.cnn_predict(augmented)    # (B,)
    dct_probs  = self.dct_predict(paths)        # (B,) -- reads files itself, as designed

    return torch.stack([clip_probs, cnn_probs, dct_probs], dim=1)  # (B, 3)

## Fusing the TopLayer with the Output Layer containing 2 hidden layers weighing the probability outputs of the 3 models.

In [ ]:
dir = "/content/drive/MyDrive/techjam/sidset_subset/test/REAL"
paths = [os.path.join(dir, f) for f in os.listdir(dir) if f.upper().endswith(('.PNG', '.JPG', '.JPEG'))]

out = top_layer(paths)          # (N, 3) in a handful of batched passes
for f, row in zip(paths, out):
    print(os.path.basename(f), row)

001000.jpg tensor([1.0777e-03, 1.0562e-04, 1.2782e-08])
001001.jpg tensor([1.3006e-05, 1.9858e-02, 1.4075e-11])
001002.jpg tensor([0.0062, 0.3287, 0.4550])
001003.jpg tensor([1.7517e-04, 2.6710e-05, 2.4261e-06])
001004.jpg tensor([1.1087e-02, 2.8794e-04, 3.1993e-08])
001005.jpg tensor([1.1149e-06, 1.0859e-10, 8.4126e-07])
001006.jpg tensor([1.7233e-01, 1.4038e-05, 5.7436e-02])
001007.jpg tensor([2.8834e-08, 2.8764e-08, 9.0785e-08])
001008.jpg tensor([0.2605, 0.0785, 0.1850])
001009.jpg tensor([3.4346e-06, 8.8646e-10, 2.5623e-06])
001010.jpg tensor([1.2353e-03, 3.1607e-03, 2.7194e-06])
001011.jpg tensor([8.8804e-05, 3.6423e-05, 1.3646e-06])
001012.jpg tensor([9.5727e-03, 3.9844e-04, 1.2015e-15])
001013.jpg tensor([8.8200e-05, 2.7765e-09, 5.2186e-05])
001014.jpg tensor([9.8388e-05, 2.3587e-05, 3.8838e-04])
001015.jpg tensor([5.1333e-06, 2.4975e-03, 1.4293e-05])
001016.jpg tensor([6.3042e-07, 2.3667e-06, 1.1731e-04])
001017.jpg tensor([6.8730e-02, 3.0202e-03, 5.5567e-06])
001018.jpg tenso

KeyboardInterrupt: 

In [196]:
def predict_paths(detector, paths, batch_size=64):
    outs = []
    batch_count = 1
    total_batches = math.ceil(len(paths) / batch_size)

    for i in range(0, len(paths), batch_size):
        print(f'Batch {batch_count}/{total_batches}')
        outs.append(detector(paths[i:i+batch_size]))
        batch_count += 1

    return torch.cat(outs, dim=0)   # (N, 3)

## Training the Top Layer + Bottom Layer

### Generating Dataset

In [221]:
real_train_dir = "/content/drive/MyDrive/techjam/sidset_subset/train/REAL"
fake_train_dir = "/content/drive/MyDrive/techjam/sidset_subset/train/FAKE"
real_files = [os.path.join(real_train_dir, f) for f in os.listdir(real_train_dir) if f.upper().endswith(('.PNG', '.JPG', '.JPEG'))][:3000]
fake_files = [os.path.join(fake_train_dir, f) for f in os.listdir(fake_train_dir) if f.upper().endswith(('.PNG', '.JPG', '.JPEG'))][:3000]

random.shuffle(real_files)
random.shuffle(fake_files)

len(real_files), len(fake_files)

(3000, 3000)

In [222]:
train_files = real_files + fake_files

train_labels = [0.0] * len(real_files) + [1.0] * len(fake_files)  # 0 is real, 1 is AI

combined = list(zip(train_files, train_labels))
random.shuffle(combined)
train_files, train_labels = map(list, zip(*combined))

print(f"Train set: {len(real_files)} real, {len(fake_files)} fake, {len(train_files)} total")

Train set: 3000 real, 3000 fake, 6000 total


In [223]:
top_layer = TopLayer()

In [224]:
x_probs = predict_paths(top_layer, train_files)
x_probs.shape

Batch 1/94
Batch 2/94
Batch 3/94
Batch 4/94
Batch 5/94
Batch 6/94
Batch 7/94
Batch 8/94
Batch 9/94
Batch 10/94
Batch 11/94
Batch 12/94
Batch 13/94
Batch 14/94
Batch 15/94
Batch 16/94
Batch 17/94
Batch 18/94
Batch 19/94
Batch 20/94
Batch 21/94
Batch 22/94
Batch 23/94
Batch 24/94
Batch 25/94
Batch 26/94
Batch 27/94
Batch 28/94
Batch 29/94
Batch 30/94
Batch 31/94
Batch 32/94
Batch 33/94
Batch 34/94
Batch 35/94
Batch 36/94
Batch 37/94
Batch 38/94
Batch 39/94
Batch 40/94
Batch 41/94
Batch 42/94
Batch 43/94
Batch 44/94
Batch 45/94
Batch 46/94
Batch 47/94
Batch 48/94
Batch 49/94
Batch 50/94
Batch 51/94
Batch 52/94
Batch 53/94
Batch 54/94
Batch 55/94
Batch 56/94
Batch 57/94
Batch 58/94
Batch 59/94
Batch 60/94
Batch 61/94
Batch 62/94
Batch 63/94
Batch 64/94
Batch 65/94
Batch 66/94
Batch 67/94
Batch 68/94
Batch 69/94
Batch 70/94
Batch 71/94
Batch 72/94
Batch 73/94
Batch 74/94
Batch 75/94
Batch 76/94
Batch 77/94
Batch 78/94
Batch 79/94
Batch 80/94
Batch 81/94
Batch 82/94
Batch 83/94
Batch 84/94
B

torch.Size([6000, 3])

In [227]:
n = random.randint(0, 5999)
x_probs[n], train_labels[n]

(tensor([1.0000, 1.0000, 1.0000]), 1.0)

In [250]:
class BottomLayer(nn.Module):
  def __init__(self, N_features=3, hidden=1):
    super().__init__()

    self.output = nn.Sequential(
        nn.Linear(3, 1),
        nn.ReLU(),
        nn.Linear(1, 1),
        nn.Sigmoid()
    )

  def forward(self, probs):
    return self.output(probs).squeeze(-1) # (B,) probabilities


In [253]:
import torch.optim as optim
import copy

# 1. Initializing Model
bottom_layer = BottomLayer()

In [254]:

best_loss = float('inf')
best_model = None
SAVE_PATH = "/content/drive/MyDrive/techjam/bottom_layer_best.pt"

# 2. Loss + optimizer
#    self.output ends in nn.Sigmoid(), so its output is already a probability --
#    use BCELoss (not BCEWithLogitsLoss, which expects raw logits)
criterion = nn.BCELoss()
optimizer = optim.Adam(bottom_layer.parameters(), lr=1e-3)

# 3. Training loop
EPOCHS = 50
BATCH_SIZE = 24

for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    permutation = torch.randperm(len(train_files))
    epoch_loss = 0.0
    n_batches = 0
    y_true = []
    y_preds = []

    for start in range(0, len(x_probs), BATCH_SIZE):
        idx = permutation[start:start + BATCH_SIZE].tolist()
        batch_paths = torch.stack([x_probs[i] for i in idx])
        batch_labels = torch.tensor([train_labels[i] for i in idx], dtype=torch.float32)

        optimizer.zero_grad()
        preds = bottom_layer(batch_paths)     # (B,) probabilities
        y_preds.extend(preds.tolist())
        y_true.extend(batch_labels.tolist())
        loss = criterion(preds, batch_labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    y_preds_hard = [1.0 if p >= 0.5 else 0.0 for p in y_preds]
    acc = accuracy_score(y_true, y_preds_hard)
    print(f"Epoch [{epoch+1}/{EPOCHS}] avg loss: {avg_loss:.4f} accuracy: {acc:.4f}")

    # save whenever this epoch beats the best loss seen so far
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_model = copy.deepcopy(bottom_layer)
        print(f"  -> new best (loss={best_loss:.4f}), checkpointed")

torch.save(best_model, SAVE_PATH)   # <-- saves the full object (architecture + weights), pickled
print(f"Saved best bottom_layer (loss={best_loss:.4f}) to {SAVE_PATH}")

Epoch 1/50
Epoch [1/50] avg loss: 0.7346 accuracy: 0.5000
  -> new best (loss=0.7346), checkpointed
Epoch 2/50
Epoch [2/50] avg loss: 0.6836 accuracy: 0.5000
  -> new best (loss=0.6836), checkpointed
Epoch 3/50
Epoch [3/50] avg loss: 0.5549 accuracy: 0.5000
  -> new best (loss=0.5549), checkpointed
Epoch 4/50
Epoch [4/50] avg loss: 0.4400 accuracy: 0.7930
  -> new best (loss=0.4400), checkpointed
Epoch 5/50
Epoch [5/50] avg loss: 0.3509 accuracy: 0.9930
  -> new best (loss=0.3509), checkpointed
Epoch 6/50
Epoch [6/50] avg loss: 0.2835 accuracy: 0.9955
  -> new best (loss=0.2835), checkpointed
Epoch 7/50
Epoch [7/50] avg loss: 0.2324 accuracy: 0.9968
  -> new best (loss=0.2324), checkpointed
Epoch 8/50
Epoch [8/50] avg loss: 0.1929 accuracy: 0.9980
  -> new best (loss=0.1929), checkpointed
Epoch 9/50
Epoch [9/50] avg loss: 0.1620 accuracy: 0.9980
  -> new best (loss=0.1620), checkpointed
Epoch 10/50
Epoch [10/50] avg loss: 0.1373 accuracy: 0.9982
  -> new best (loss=0.1373), checkpointe

# THE PIPELINE
Attached below is all the code needed for the Pipeline

In [255]:
from PIL import Image
import joblib, torch, open_clip, torch.nn as nn, torch.nn.functional as F
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
import pandas as pd

In [ ]:
from scipy.fft import dctn

def stage1_2_block_dct_from_array(bgr):
    """
    Stage 1 (Ingestion) + Stage 2 (Mathematical prep) on an already-loaded BGR array.
    Returns: (h_blocks, w_blocks, 8, 8) array of Y-channel block DCT coefficients.
    """
    ycbcr = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)  # Y, Cr, Cb
    y = ycbcr[:, :, 0].astype(np.float32) - 128.0   # center like JPEG does

    h, w = y.shape
    hb, wb = h // 8, w // 8
    y = y[:hb * 8, :wb * 8]  # crop any leftover rows/cols that don't fill a full 8x8 block

    blocks = y.reshape(hb, 8, wb, 8).transpose(0, 2, 1, 3)
    blocks = dctn(blocks, type=2, norm='ortho', axes=(-2, -1))

    return blocks.astype(np.float32)


def stage3_forensic_scan(blocks):
    """
    blocks: (hb, wb, 8, 8) block DCT coefficients.
    Returns (coeff_stats, [upsampling_score, double_compress_score]).
    """
    hb, wb, _, _ = blocks.shape
    flat = blocks.reshape(hb, wb, 64)

    coeff_stats = flat.mean(axis=(0, 1))

    dc_grid = flat[:, :, 0]
    fft_mag = np.abs(np.fft.fft2(dc_grid))
    fft_mag[0, 0] = 0
    upsampling_score = fft_mag.max() / (fft_mag.mean() + 1e-8)

    ac1 = flat[:, :, 1].flatten()
    hist, _ = np.histogram(ac1, bins=16)
    hist_fft = np.abs(np.fft.fft(hist.astype(np.float32)))
    hist_fft[0] = 0
    double_compress_score = hist_fft.max() / (hist_fft.mean() + 1e-8)

    return coeff_stats, np.array([upsampling_score, double_compress_score], dtype=np.float32)

class CoeffResNet(nn.Module):

  def __init__(self, in_ch=64, emb_dim=128):
      super().__init__()
      self.conv1 = nn.Conv2d(in_ch, 128, 3, padding=1)
      self.bn1 = nn.BatchNorm2d(128)
      self.conv2 = nn.Conv2d(128, emb_dim, 3, padding=1)
      self.bn2 = nn.BatchNorm2d(emb_dim)
      self.head = nn.Linear(emb_dim, 1)

  def forward(self, x, return_embedding=False):
      x = F.relu(self.bn1(self.conv1(x)))
      x = F.relu(self.bn2(self.conv2(x)))
      emb = F.adaptive_avg_pool2d(x, 1).flatten(1)  # (B, emb_dim)
      if return_embedding:
          return emb
      return self.head(emb).squeeze(-1)

In [ ]:
class TopLayer(nn.Module):
  def __init__(self,
               clip_model="/content/drive/MyDrive/techjam/standard_v2.joblib",
               cnn_model="/content/drive/MyDrive/techjam/efficientnet_b0_detector-2.pt",
               dct_model="/content/drive/MyDrive/techjam/dct_logreg.joblib",
               dct_coeff_model="/content/drive/MyDrive/techjam/dct_coeff_net.pth"):

    super().__init__()

    self.CHECKPOINT_DIR = "/content/drive/MyDrive/techjam"
    self.DCT_SIZE = 256  # must match the resize used in ForensicDataset during DCT-model training

    self.device = "cuda" if torch.cuda.is_available() else "cpu"

    self.clip = joblib.load(clip_model)
    self.cnn = self.build_efficientnet(freeze_until_last_n_blocks=1).eval()

    # DCT pipeline: sklearn clf (StandardScaler + LogisticRegression) + CoeffResNet embedding CNN
    self.dct = joblib.load(dct_model)
    self.dct_coeff_net = CoeffResNet(in_ch=64).to(self.device)
    self.dct_coeff_net.load_state_dict(torch.load(dct_coeff_model, map_location=self.device))
    self.dct_coeff_net.eval()

    self.model, _, self.clip_preprocess = open_clip.create_model_and_transforms(self.clip["clip_model"], pretrained=self.clip["clip_pretrained"])
    self.model = self.model.to(self.device).eval()

    self.cnn_transform = A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

  def build_efficientnet(self, freeze_until_last_n_blocks=1):

    checkpoint = torch.load(
        os.path.join(self.CHECKPOINT_DIR, "efficientnet_b0_detector-2.pt"),
        map_location=self.device,
        weights_only=False,
    )

    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=1)

    for param in model.parameters():
        param.requires_grad = False
    for stage in model.blocks[-freeze_until_last_n_blocks:]:
        for param in stage.parameters():
            param.requires_grad = True
    for module in [model.conv_head, model.bn2, model.classifier]:
        for param in module.parameters():
            param.requires_grad = True
    model = model.to(self.device)

    model.load_state_dict(checkpoint["state_dict"])

    return model

  @torch.no_grad()
  def clip_predict(self, imgs, which="robust"):
      '''Batched: imgs is a list of HxWx3 uint8 arrays. Returns (B,) tensor of P(AI).'''
      if isinstance(imgs, np.ndarray) and imgs.ndim == 3:  # single-image fallback
          imgs = [imgs]
      # preprocess each (resizes to CLIP's fixed input) then stack -> one encoder pass
      batch = torch.stack([self.clip_preprocess(Image.fromarray(a)) for a in imgs]).to(self.device)
      f = F.normalize(self.model.encode_image(batch).float(), dim=-1).cpu().numpy()  # (B, D)
      probs = self.clip["probes"][which].predict_proba(f)[:, 1]                       # vectorised over B
      return torch.tensor(probs, dtype=torch.float32)


  def cnn_predict(self, imgs):
      '''Batched: imgs is a list of HxWx3 uint8 arrays. Returns (B,) tensor of P(AI).'''
      if isinstance(imgs, np.ndarray) and imgs.ndim == 3:
          imgs = [imgs]
      self.cnn.eval()
      with torch.no_grad():
          # A.Resize(224,224) makes every image the same shape -> stackable
          batch = torch.stack([self.cnn_transform(image=a)["image"] for a in imgs]).to(self.device)
          logits = self.cnn(batch).squeeze(1)          # one forward pass for all B
          probs = torch.sigmoid(logits).detach().cpu()
      return probs                                      # (B,)


  def dct_predict(self, paths):
      '''Batched: paths is a list of file-path strings. Returns (B,) tensor of P(AI).'''
      if isinstance(paths, str):
          paths = [paths]
      coeff_mats, forensic_list = [], []
      for p in paths:
          bgr = cv2.imread(p, cv2.IMREAD_COLOR)
          bgr = cv2.resize(bgr, (self.DCT_SIZE, self.DCT_SIZE), interpolation=cv2.INTER_AREA)
          blocks = stage1_2_block_dct_from_array(bgr)              # (hb, wb, 8, 8)
          _, forensic_scalars = stage3_forensic_scan(blocks)      # (2,)
          hb, wb, _, _ = blocks.shape
          coeff_mats.append(torch.tensor(blocks.reshape(hb, wb, 64)).permute(2, 0, 1).float())
          forensic_list.append(forensic_scalars)
      # fixed DCT_SIZE => every coeff map is (64, 32, 32) => stackable into one CNN pass
      batch = torch.stack(coeff_mats).to(self.device)             # (B, 64, 32, 32)
      with torch.no_grad():
          cnn_emb = self.dct_coeff_net(batch, return_embedding=True).cpu().numpy()  # (B, 128)
      feat = np.concatenate([cnn_emb, np.stack(forensic_list)], axis=1)             # (B, 130)
      probs = self.dct.predict_proba(feat)[:, 1]                                    # vectorised
      return torch.tensor(probs, dtype=torch.float32)

  def forward(self, paths):
    '''Accepts a single path (str) or a list of paths. Returns (B, 3) tensor:
       columns are [clip_prob, cnn_prob, dct_prob].'''
    if isinstance(paths, str):
        paths = [paths]

    img_arrs  = [np.array(Image.open(p).convert("RGB")) for p in paths]
    augmented = [random_augment(a) for a in img_arrs]   # per-image random aug, as before

    clip_probs = self.clip_predict(augmented)   # (B,)
    cnn_probs  = self.cnn_predict(augmented)    # (B,)
    dct_probs  = self.dct_predict(paths)        # (B,) -- reads files itself, as designed

    return torch.stack([clip_probs, cnn_probs, dct_probs], dim=1)  # (B, 3)

In [ ]:
def predict_paths(detector, paths, batch_size=64):
    outs = []
    batch_count = 1
    total_batches = math.ceil(len(paths) / batch_size)

    for i in range(0, len(paths), batch_size):
        print(f'Batch {batch_count}/{total_batches}')
        outs.append(detector(paths[i:i+batch_size]))
        batch_count += 1

    return torch.cat(outs, dim=0)   # (N, 3)

In [ ]:
class BottomLayer(nn.Module):
  def __init__(self, N_features=3, hidden=1):
    super().__init__()

    self.output = nn.Sequential(
        nn.Linear(N_features, hidden),
        nn.ReLU(),
        nn.Linear(1, 1),
        nn.Sigmoid()
    )

  def forward(self, probs):
    return self.output(probs).squeeze(-1) # (B,) probabilities


In [315]:
import json
from google.colab import files

class AIPipeline(nn.Module):
  def __init__(self,
               clip_model="/content/drive/MyDrive/techjam/standard_v2.joblib",
               cnn_model="/content/drive/MyDrive/techjam/efficientnet_b0_detector-2.pt",
               dct_model="/content/drive/MyDrive/techjam/dct_logreg.joblib",
               dct_coeff_model="/content/drive/MyDrive/techjam/dct_coeff_net.pth",
               fusion_model="/content/drive/MyDrive/techjam/bottom_layer_best.pt",
               N_features=3,
               hidden=1):

    super().__init__()

    self.device = "cuda" if torch.cuda.is_available() else "cpu"
    self.top_layer = TopLayer(clip_model, cnn_model, dct_model, dct_coeff_model)
    self.bottom_layer = torch.load(fusion_model, weights_only=False, map_location=self.device)
    self.bottom_layer.eval()

  def predict_paths(self, paths, batch_size=64):
    outs = []
    batch_count = 1
    total_batches = math.ceil(len(paths) / batch_size)

    for i in range(0, len(paths), batch_size):
        print(f'Batch {batch_count}/{total_batches}')
        outs.append(self.top_layer(paths[i:i+batch_size]))
        batch_count += 1

    return torch.cat(outs, dim=0)

  def forward(self, paths):
    if type(paths) != list:
      paths = paths.tolist()

    self.probs = self.predict_paths(paths)
    y_probs =  self.bottom_layer(self.probs).tolist()
    json_prob = [["image_path", "pred"]]
    json_prob.extend([[path, prob] for path, prob in zip(paths, y_probs)])

    with open("output.json", "w") as file:
      json.dump(json_prob, file, indent=4)

    files.download("output.json")
    print("Results written to output.json")
    return y_probs

## AI Pipeline Demo

In [301]:
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import torch # Ensure torch is imported for device check

def load_valset(dir, realfolder, fakefolder, n_images=100):
  realpath = os.path.join(dir, realfolder)
  fakepath = os.path.join(dir, fakefolder)

  real_files = os.listdir(realpath)
  fake_files = os.listdir(fakepath)

  real_files = [(os.path.join(realpath, real_file), 0) for real_file in real_files]
  fake_files = [(os.path.join(fakepath, fake_file), 1) for fake_file in fake_files]
  random.shuffle(real_files)
  random.shuffle(fake_files)
  valset = real_files[:n_images] + fake_files[:n_images]

  random.shuffle(valset)

  return [img for img, label in valset], [label for img, label in valset]

def validate(model, x, y_true):
  criterion = nn.BCEWithLogitsLoss()
  y_pred = []

  for img in x:

    img_tensor = Image.open(img).convert("RGB")

    # Ensure img_tensor is on the same device as the model
    # It's safer to get the model's device directly
    model_device = next(model.parameters()).device
    img_tensor = TF.to_tensor(img_tensor).unsqueeze(0).to(model_device)

    with torch.no_grad():
      logit = model(img_tensor)[0].squeeze()

    y_pred.append(logit)

  y_pred = torch.tensor(y_pred)

  y_pred = torch.sigmoid(y_pred).detach().cpu().numpy()
  y_pred = y_pred.tolist()
  y_pred = [1 if pred > 0.5 else 0 for pred in y_pred]

  # cm = confusion_matrix(y_true, y_pred)

  # # Display with custom text labels

  # disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
  # disp.plot(cmap=plt.cm.Blues)

  # plt.show()
  print(y_pred)
  return accuracy_score(y_true, y_pred)

In [312]:
x, y = load_valset("/content/drive/MyDrive/techjam/ValidationDataset", "val2017", "dalle3/2023110215025084768300d30fc34f", 20)

In [316]:
ai_model = AIPipeline()

In [317]:
y_pred = ai_model(x)

Batch 1/1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Results written to output.json
